In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Train/Test
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

# MLflow
import mlflow
import mlflow.sklearn
import mlflow.xgboost

# Optuna (para después)
import optuna
from optuna.samplers import TPESampler

# Otros
from datetime import datetime

# ── Constantes globales ──
RANDOM_STATE = 42
TEST_SIZE    = 0.2

In [2]:
# Cargar dataset
df = pd.read_csv("../data/raw/stroke_dataset.csv")

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"\nPrimeras filas:")
print(df.head(3))

Dataset cargado: 4981 filas, 11 columnas

Primeras filas:
   gender   age  hypertension  heart_disease ever_married work_type  \
0    Male  67.0             0              1          Yes   Private   
1    Male  80.0             0              1          Yes   Private   
2  Female  49.0             0              0          Yes   Private   

  Residence_type  avg_glucose_level   bmi   smoking_status  stroke  
0          Urban             228.69  36.6  formerly smoked       1  
1          Rural             105.92  32.5     never smoked       1  
2          Urban             171.23  34.4           smokes       1  


In [3]:
import os

# MLflow está en la raíz del proyecto
mlruns_path = os.path.abspath("../mlruns")

# Decirle a MLflow dónde guardar
mlflow.set_tracking_uri(f"file:///{mlruns_path}")

print(f"✓ MLflow usando carpeta compartida: {mlruns_path}")

✓ MLflow usando carpeta compartida: c:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\mlruns


In [4]:
def get_dataset(version="full"):
    """
    Obtiene el dataset en diferentes versiones
    
    version="full": todos los registros + columna 'group'
    version="adults": solo mayores de 16 años (sin columna 'group')
    """
    
    df_copy = df.copy()
    
    # LIMPIEZA 1: menores de 16 con smoking_status Unknown → not_applied
    df_copy.loc[(df_copy['age'] < 17) & (df_copy['smoking_status'] == 'Unknown'), 'smoking_status'] = 'not_applied'
    
    # LIMPIEZA 2: work_type='children' → 'not_applied'
    df_copy.loc[df_copy['work_type'] == 'children', 'work_type'] = 'not_applied'
    
    if version == "full":
        # Crear columna group: 'children' o 'adults'
        df_copy['group'] = np.where(df_copy['age'] < 17, 'children', 'adults')
        return df_copy
    
    elif version == "adults":
        # Filtrar solo adultos (age > 16)
        df_copy = df_copy[df_copy['age'] > 16].copy()
        # NO añadimos columna group en esta versión
        return df_copy

##  Función train_model_with_pipeline()

### ¿Qué hace?
Función que **entrena automáticamente un modelo** usando un Pipeline.

### Los 3 pasos internos:
1. **OneHot Encoding**: Convierte texto (gender, work_type, etc.) en números (0/1)
2. **SMOTE**: Crea datos sintéticos de la clase minoritaria (stroke=1) SOLO en train
3. **Entrenar modelo**: Entrena el modelo (Logistic o XGBoost)

### Entrada (Input):
- `X_train, X_test`: Features de entrenamiento y test
- `y_train, y_test`: Targets de entrenamiento y test
- `model`: El modelo a entrenar (ej: LogisticRegression())
- `model_name`: Nombre del modelo para MLflow (ej: "LogisticRegression")
- `dataset_version`: "full" o "adults"

### Salida (Output):
- `pipeline`: El modelo entrenado + procesamiento
- `acc, prec, rec, f1, auc`: Las 5 métricas principales

### ¿Por qué Pipeline?
Garantiza que los datos pasen por los MISMOS pasos en train Y test.
SMOTE solo actúa en entrenamiento, no en predicción (evita data leakage).

### Ejemplo de uso:
```python
pipeline, acc, prec, rec, f1, auc = train_model_with_pipeline(
    X_train, X_test, y_train, y_test,
    LogisticRegression(),
    "LogisticRegression",
    "full"
)
```

In [5]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# FUNCIÓN 1: Para Logistic Regression (CON Scaling)
def train_logistic_with_pipeline(X_train, X_test, y_train, y_test, model, dataset_version="full"):
    """
    Entrena Logistic Regression con Pipeline (OneHot + Scaling + SMOTE + Modelo)
    """
    
    cat_cols = X_train.select_dtypes(include='object').columns.tolist()
    num_cols = X_train.select_dtypes(exclude='object').columns.tolist()
    
    # Preprocesador CON Scaling para numéricas
    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)  #  Scaling aquí
    ])
    
    pipeline = Pipeline([
        ('prep', preprocessor),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', model)
    ])
    
    print(f"\n=== Logistic Regression | {dataset_version} ===")
    pipeline.fit(X_train, y_train)
    
    # y_pred = pipeline.predict(X_test)
    # y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Predicciones en TRAIN y TEST
    y_pred_train = pipeline.predict(X_train)
    y_pred_test = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Métricas en TRAIN
    acc_train = accuracy_score(y_train, y_pred_train)
    f1_train = f1_score(y_train, y_pred_train, average='weighted')

    # Métricas en TEST
    acc_test = accuracy_score(y_test, y_pred_test)
    f1_test = f1_score(y_test, y_pred_test, average='weighted')

    # Diferencias (para detectar overfitting)
    diff_acc = abs(acc_train - acc_test)
    diff_f1 = abs(f1_train - f1_test)

    print(f"Train Acc: {acc_train:.4f} | Test Acc: {acc_test:.4f} | Diff: {diff_acc:.4f}")
    print(f"Train F1:  {f1_train:.4f} | Test F1:  {f1_test:.4f} | Diff: {diff_f1:.4f}")

    if diff_acc < 0.05:
        print("✓ Sin overfitting")
    else:
        print("⚠ Posible overfitting")
        
    
    acc = acc_test  # Ya está calculado
    prec = precision_score(y_test, y_pred_test, average='weighted')
    rec = recall_score(y_test, y_pred_test, average='weighted')
    f1 = f1_test  # Ya está calculado
    auc = roc_auc_score(y_test, y_prob)
        
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    
    return pipeline, acc, prec, rec, f1, auc, diff_acc, diff_f1


In [6]:
# FUNCIÓN 2: Para XGBoost (SIN Scaling)
def train_xgboost_with_pipeline(X_train, X_test, y_train, y_test, model, dataset_version="full"):
    """
    Entrena XGBoost con Pipeline (OneHot + SMOTE + Modelo, sin Scaling)
    """
    
    cat_cols = X_train.select_dtypes(include='object').columns.tolist()
    num_cols = X_train.select_dtypes(exclude='object').columns.tolist()
    
    # Preprocesador SIN Scaling para numéricas
    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)  # ✗ Sin scaling
    ])
    
    pipeline = Pipeline([
        ('prep', preprocessor),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', model)
    ])
    
    print(f"\n=== XGBoost | {dataset_version} ===")
    pipeline.fit(X_train, y_train)
    
    # y_pred = pipeline.predict(X_test)
    # y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    # Predicciones en TRAIN y TEST
    y_pred_train = pipeline.predict(X_train)
    y_pred_test = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Métricas en TRAIN
    acc_train = accuracy_score(y_train, y_pred_train)
    f1_train = f1_score(y_train, y_pred_train, average='weighted')

    # Métricas en TEST
    acc_test = accuracy_score(y_test, y_pred_test)
    f1_test = f1_score(y_test, y_pred_test, average='weighted')

    # Diferencias (para detectar overfitting)
    diff_acc = abs(acc_train - acc_test)
    diff_f1 = abs(f1_train - f1_test)

    print(f"Train Acc: {acc_train:.4f} | Test Acc: {acc_test:.4f} | Diff: {diff_acc:.4f}")
    print(f"Train F1:  {f1_train:.4f} | Test F1:  {f1_test:.4f} | Diff: {diff_f1:.4f}")

    if diff_acc < 0.05:
        print("✓ Sin overfitting")
    else:
        print("⚠ Posible overfitting")
        
    
    acc = acc_test  # Ya está calculado
    prec = precision_score(y_test, y_pred_test, average='weighted')
    rec = recall_score(y_test, y_pred_test, average='weighted')
    f1 = f1_test  # Ya está calculado
    auc = roc_auc_score(y_test, y_prob)
        
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    
    return pipeline, acc, prec, rec, f1, auc, diff_acc, diff_f1

### CREAMOS EL MODELO REGRESION LOGISTICA

In [7]:
# Crear el modelo Logistic Regression CON Ridge Loss
log_model = LogisticRegression(
    class_weight='balanced',
    penalty='l2',              # ← Ridge Loss (L2)
    C=1.0,                     # Parámetro de regularización (menor C = más regularización)
    max_iter=1000,
    random_state=RANDOM_STATE
)

print("Modelo Logistic Regression (con Ridge Loss L2) creado ✓")

Modelo Logistic Regression (con Ridge Loss L2) creado ✓


### PREPARAMOS PARA ENTRENARCON REGRESION LOGISTICA CON EL DATASET FULL (el que tiene columnas separadas niños y adultos)

In [8]:
# Obtener dataset FULL (con niños)
df_full = get_dataset(version="full")

print(f"Dataset FULL cargado: {df_full.shape[0]} filas")
print(f"Columnas: {df_full.columns.tolist()}")

# Separar X e y
X_full = df_full.drop('stroke', axis=1)
y_full = df_full['stroke']

# Split train/test
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_full
)

print(f"\nDataset FULL dividido:")
print(f"  X_train: {X_train_full.shape}")
print(f"  X_test: {X_test_full.shape}")

Dataset FULL cargado: 4981 filas
Columnas: ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke', 'group']

Dataset FULL dividido:
  X_train: (3984, 11)
  X_test: (997, 11)


In [9]:
# Entrenar Logistic Regression con dataset FULL
pipeline_log_full, acc_log_full, prec_log_full, rec_log_full, f1_log_full, auc_log_full, diff_acc_log_full, diff_f1_log_full = train_logistic_with_pipeline(
    X_train_full, X_test_full, y_train_full, y_test_full,
    log_model,
    dataset_version="full"
)

print(f"\n✓ Logistic Regression (FULL) entrenado")
print(f"  Overfitting - Diff Accuracy: {diff_acc_log_full:.4f}")
print(f"  Overfitting - Diff F1: {diff_f1_log_full:.4f}")


=== Logistic Regression | full ===
Train Acc: 0.7432 | Test Acc: 0.7482 | Diff: 0.0050
Train F1:  0.8154 | Test F1:  0.8185 | Diff: 0.0031
✓ Sin overfitting
Accuracy: 0.7482 | Precision: 0.9423 | Recall: 0.7482 | F1: 0.8185 | AUC: 0.8372

✓ Logistic Regression (FULL) entrenado
  Overfitting - Diff Accuracy: 0.0050
  Overfitting - Diff F1: 0.0031


### HACEMOS MATRIZ DE CONFUSION

In [10]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns
import os
from pathlib import Path

# Crear carpeta assets si no existe
BASE_DIR = Path().resolve().parent  # Sube de notebooks/ a raíz
assets_dir = BASE_DIR / "assets"
assets_dir.mkdir(exist_ok=True)  # Crea si no existe

# Hacer predicción
y_pred_log_full = pipeline_log_full.predict(X_test_full)

# Matriz de confusión
cm = confusion_matrix(y_test_full, y_pred_log_full)

# Gráfico
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],
            yticklabels=['No Stroke', 'Stroke'])
plt.title('Matriz de Confusión - Logistic Regression (FULL)')
plt.ylabel('Actual')
plt.xlabel('Predicción')
plt.tight_layout()

# Guardar en assets/
cm_path = os.path.join(assets_dir, "confusion_matrix_lr_full_v2.png")
plt.savefig(cm_path, dpi=100, bbox_inches='tight')
plt.close()

print(f"✓ Matriz guardada en: {cm_path}")

✓ Matriz guardada en: C:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\assets\confusion_matrix_lr_full_v2.png


### USAMOS MLFLOW

In [11]:
# Registrar en MLflow - Logistic Regression FULL
mlflow.set_experiment("stroke_project")

with mlflow.start_run(run_name="LogisticRegression_full_v2"):
    
    # Parámetros
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("dataset", "full")
    mlflow.log_param("penalty", "l2")
    mlflow.log_param("C", 1.0)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("author", "Gema")
    
    # Métricas 
    mlflow.log_metric("accuracy", round(acc_log_full, 4))
    mlflow.log_metric("precision", round(prec_log_full, 4))
    mlflow.log_metric("recall", round(rec_log_full, 4))
    mlflow.log_metric("f1", round(f1_log_full, 4))
    mlflow.log_metric("auc", round(auc_log_full, 4))
    
    # Métricas OVERFITTING (NUEVO)
    mlflow.log_metric("overfitting_diff_accuracy", round(diff_acc_log_full, 4))
    mlflow.log_metric("overfitting_diff_f1", round(diff_f1_log_full, 4))
    
    # Gráfico (matriz de confusión)
    mlflow.log_artifact(cm_path, artifact_path="confusion_matrices")
    
    # Modelo
    # mlflow.sklearn.log_model(pipeline_log_full, "modelo_logistic") de momento no lo exportamos para no llenar el MLflow de modelos grandes.
    
    
    print("✓ LogisticRegression FULL registrado en MLflow")

mlflow.end_run()

✓ LogisticRegression FULL registrado en MLflow


### CREAMOS EL MODELO XGBOOST

In [12]:
# Crear el modelo XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    scale_pos_weight=19,  # Porque stroke es minoritaria (95% vs 5%)
    eval_metric='logloss'
)

print("Modelo XGBoost creado ✓")

Modelo XGBoost creado ✓


### PREPARAMOS PARA ENTRENARCON XGBOOST CON EL DATASET FULL (el que tiene columnas separadas niños y adultos)

In [13]:
# Obtener dataset FULL (igual que con Logistic)
df_full = get_dataset(version="full")

# Separar X e y
X_full = df_full.drop('stroke', axis=1)
y_full = df_full['stroke']

# Split train/test
X_train_full_xgb, X_test_full_xgb, y_train_full_xgb, y_test_full_xgb = train_test_split(
    X_full, y_full,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_full
)

print(f"Dataset FULL para XGBoost:")
print(f"  X_train: {X_train_full_xgb.shape}")
print(f"  X_test: {X_test_full_xgb.shape}")

Dataset FULL para XGBoost:
  X_train: (3984, 11)
  X_test: (997, 11)


In [14]:
pipeline_xgb_full, acc_xgb_full, prec_xgb_full, rec_xgb_full, f1_xgb_full, auc_xgb_full, diff_acc_xgb_full, diff_f1_xgb_full = train_xgboost_with_pipeline(
    X_train_full_xgb, X_test_full_xgb, y_train_full_xgb, y_test_full_xgb,
    xgb_model,
    dataset_version="full"
)

print(f"\n✓ XGBoost (FULL) entrenado")
print(f"  Overfitting - Diff Accuracy: {diff_acc_xgb_full:.4f}")
print(f"  Overfitting - Diff F1: {diff_f1_xgb_full:.4f}")


=== XGBoost | full ===
Train Acc: 0.9352 | Test Acc: 0.8786 | Diff: 0.0566
Train F1:  0.9469 | Test F1:  0.8992 | Diff: 0.0477
⚠ Posible overfitting
Accuracy: 0.8786 | Precision: 0.9254 | Recall: 0.8786 | F1: 0.8992 | AUC: 0.8123

✓ XGBoost (FULL) entrenado
  Overfitting - Diff Accuracy: 0.0566
  Overfitting - Diff F1: 0.0477


### HACEMOS MATRIZ DE CONFUSION

In [15]:
# Hacer predicción
y_pred_xgb_full = pipeline_xgb_full.predict(X_test_full_xgb)
# Matriz de confusión
cm_xgb_full = confusion_matrix(y_test_full_xgb, y_pred_xgb_full)
# Gráfico
plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb_full, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],
            yticklabels=['No Stroke', 'Stroke'])
plt.title('Matriz de Confusión - XGBoost (FULL)')
plt.ylabel('Actual')
plt.xlabel('Predicción')
plt.tight_layout()

# Guardar en assets/
cm_xgb_path = os.path.join(assets_dir, "confusion_matrix_xgb_full_v2.png")
plt.savefig(cm_xgb_path, dpi=100, bbox_inches='tight')
plt.close()

print(f"✓ Matriz XGBoost guardada en: {cm_xgb_path}")


✓ Matriz XGBoost guardada en: C:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\assets\confusion_matrix_xgb_full_v2.png


### REGISTRAMOS EN MLFLOW

In [16]:
# Registrar en MLflow - XGBoost FULL (métricas + gráfico)
with mlflow.start_run(run_name="XGBoost_full_v2"):
    
    # Parámetros
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("dataset", "full")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("scale_pos_weight", 19)
    mlflow.log_param("author", "Gema")  
    
    # Métricas TEST
    mlflow.log_metric("accuracy", round(acc_xgb_full, 4))
    mlflow.log_metric("precision", round(prec_xgb_full, 4))
    mlflow.log_metric("recall", round(rec_xgb_full, 4))
    mlflow.log_metric("f1", round(f1_xgb_full, 4))
    mlflow.log_metric("auc", round(auc_xgb_full, 4))
    
    # Métricas OVERFITTING 
    mlflow.log_metric("overfitting_diff_accuracy", round(diff_acc_xgb_full, 4))
    mlflow.log_metric("overfitting_diff_f1", round(diff_f1_xgb_full, 4))
    
    # Gráfico (matriz de confusión)
    mlflow.log_artifact(cm_xgb_path, artifact_path="confusion_matrices")
    
     # Modelo
    # mlflow.sklearn.log_model(pipeline_xgb_full, "modelo_xgboost") de momento no lo exportamos para no llenar el MLflow de modelos grandes.
    
    
    
    print("✓ XGBoost FULL registrado (métricas + gráfico)")

mlflow.end_run()

✓ XGBoost FULL registrado (métricas + gráfico)


### CARGAMOS DATASET ADULTOS Y PREPARAMOS PARA EL ENTRENAMIENTO

In [17]:
df_adults = get_dataset(version="adults")
print(f"Dataset ADULTS cargado: {df_adults.shape[0]} filas")
print(f"Columnas: {df_adults.columns.tolist()}")

#separar X e y
X_adults = df_adults.drop('stroke', axis=1) 
y_adults = df_adults['stroke']

# Split train/test
X_train_adults, X_test_adults, y_train_adults, y_test_adults = train_test_split(
    X_adults, y_adults,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_adults
)

print(f"\nDataset ADULTS dividido:")
print(f"  X_train: {X_train_adults.shape}") 

Dataset ADULTS cargado: 4211 filas
Columnas: ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']

Dataset ADULTS dividido:
  X_train: (3368, 10)


### ENTRENAMOS CON REGRESION LOGISTICA

In [18]:
pipeline_log_adults, acc_log_adults, prec_log_adults, rec_log_adults, f1_log_adults, auc_log_adults, diff_acc_log_adults, diff_f1_log_adults = train_logistic_with_pipeline(
    X_train_adults, X_test_adults, y_train_adults, y_test_adults,
    log_model,  
    dataset_version="adults"
)   
print(f"\n✓ Logistic Regression (ADULTS) entrenado")
print(f"  Overfitting - Diff Accuracy: {diff_acc_log_adults:.4f}")
print(f"  Overfitting - Diff F1: {diff_f1_log_adults:.4f}")


=== Logistic Regression | adults ===
Train Acc: 0.7221 | Test Acc: 0.7082 | Diff: 0.0139
Train F1:  0.7955 | Test F1:  0.7857 | Diff: 0.0098
✓ Sin overfitting
Accuracy: 0.7082 | Precision: 0.9335 | Recall: 0.7082 | F1: 0.7857 | AUC: 0.8389

✓ Logistic Regression (ADULTS) entrenado
  Overfitting - Diff Accuracy: 0.0139
  Overfitting - Diff F1: 0.0098


### HACEMOS MATRIZ DE CONFUSION

In [19]:
y_pred_log_adults = pipeline_log_adults.predict(X_test_adults)
cm_log_adults = confusion_matrix(y_test_adults, y_pred_log_adults)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_log_adults, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],
            yticklabels=['No Stroke', 'Stroke'])
plt.title('Matriz de Confusión - Logistic Regression (ADULTS)')
plt.ylabel('Actual')
plt.xlabel('Predicción')
plt.tight_layout()

# Guardar en assets/
cm_log_path = os.path.join(assets_dir, "confusion_matrix_log_adults_v2.png")
plt.savefig(cm_log_path, dpi=100, bbox_inches='tight')
plt.close()

print(f"✓ Matriz Logistic Regression guardada en: {cm_log_path}")

✓ Matriz Logistic Regression guardada en: C:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\assets\confusion_matrix_log_adults_v2.png


### REGISTRAMOS EN MLFLOW

In [20]:
with mlflow.start_run(run_name="LogisticRegression_adults_v2"):
    
    # Parámetros
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("dataset", "adults")
    mlflow.log_param("penalty", "l2")
    mlflow.log_param("C", 1.0)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("author", "Gema")  
    
    # Métricas TEST
    mlflow.log_metric("accuracy", round(acc_log_adults, 4))
    mlflow.log_metric("precision", round(prec_log_adults, 4))
    mlflow.log_metric("recall", round(rec_log_adults, 4))
    mlflow.log_metric("f1", round(f1_log_adults, 4))
    mlflow.log_metric("auc", round(auc_log_adults, 4))
    
    # Métricas OVERFITTING 
    mlflow.log_metric("overfitting_diff_accuracy", round(diff_acc_log_adults, 4))
    mlflow.log_metric("overfitting_diff_f1", round(diff_f1_log_adults, 4))
    
    # Gráfico (matriz de confusión)
    mlflow.log_artifact(cm_log_path, artifact_path="confusion_matrices")
    
     # Modelo
    # mlflow.sklearn.log_model(pipeline_log_adults, "modelo_logistic_adults") de momento no lo exportamos para no llenar el MLflow de modelos grandes.
    
    
    
    print("✓ Logistic Regression ADULTS registrado (métricas + gráfico)")

✓ Logistic Regression ADULTS registrado (métricas + gráfico)


### ENTRENAMOS CON XGBOST

In [21]:
pipeline_xgboost_adults, acc_xgb_adults, prec_xgb_adults, rec_xgb_adults, f1_xgb_adults, auc_xgb_adults, diff_acc_xgb_adults, diff_f1_xgb_adults = train_xgboost_with_pipeline(
    X_train_adults, X_test_adults, y_train_adults, y_test_adults,
    xgb_model,
    dataset_version="adults")

print(f"\n✓ XGBoost (ADULTS) entrenado")
print(f"  Overfitting - Diff Accuracy: {diff_acc_xgb_adults:.4f}")
print(f"  Overfitting - Diff F1: {diff_f1_xgb_adults:.4f}")


=== XGBoost | adults ===
Train Acc: 0.9121 | Test Acc: 0.8458 | Diff: 0.0663
Train F1:  0.9288 | Test F1:  0.8752 | Diff: 0.0536
⚠ Posible overfitting
Accuracy: 0.8458 | Precision: 0.9152 | Recall: 0.8458 | F1: 0.8752 | AUC: 0.7813

✓ XGBoost (ADULTS) entrenado
  Overfitting - Diff Accuracy: 0.0663
  Overfitting - Diff F1: 0.0536


### HACEMOS MATRIZ DE CONFUSION

In [22]:
y_pred_xgb_adults = pipeline_xgboost_adults.predict(X_test_adults)
cm_xgb_adults = confusion_matrix(y_test_adults, y_pred_xgb_adults)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb_adults, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Stroke', 'Stroke'],
            yticklabels=['No Stroke', 'Stroke'])
plt.title('Matriz de Confusión - XBoost (ADULTS)')
plt.ylabel('Actual')
plt.xlabel('Predicción')
plt.tight_layout()

# Guardar en assets/
cm_xgb_path = os.path.join(assets_dir, "confusion_matrix_xgb_adults_v2.png")
plt.savefig(cm_xgb_path, dpi=100, bbox_inches='tight')
plt.close()

print(f"✓ Matriz XBoost guardada en: {cm_xgb_path}")



✓ Matriz XBoost guardada en: C:\Users\gemit\Desktop\factoria-ia\PROYECTOS\Project_8_Equipo1\assets\confusion_matrix_xgb_adults_v2.png


### REGISTRAMOS EN MLFLOW

In [23]:
with mlflow.start_run(run_name="XGBoost_adults_v2"):
    # Parámetros
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("dataset", "adults")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("scale_pos_weight", 19)
    mlflow.log_param("author", "Gema")  
    
    # Métricas TEST
    mlflow.log_metric("accuracy", round(acc_xgb_adults, 4))
    mlflow.log_metric("precision", round(prec_xgb_adults, 4))
    mlflow.log_metric("recall", round(rec_xgb_adults, 4))
    mlflow.log_metric("f1", round(f1_xgb_adults, 4))
    mlflow.log_metric("auc", round(auc_xgb_adults, 4))
    
    # Métricas OVERFITTING 
    mlflow.log_metric("overfitting_diff_accuracy", round(diff_acc_xgb_adults, 4))
    mlflow.log_metric("overfitting_diff_f1", round(diff_f1_xgb_adults, 4))
    
    # Gráfico (matriz de confusión)
    mlflow.log_artifact(cm_xgb_path, artifact_path="confusion_matrices")
    
    #Modelo
    mlflow.sklearn.log_model(pipeline_xgboost_adults, "modelo_xgboost_adults") #de momento no lo exportamos para no llenar el MLflow de modelos grandes.
    
    
    
    print("✓ XGBoost ADULTS registrado (métricas + gráfico)")

2026/04/20 11:03:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 11:03:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/20 11:03:21 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


✓ XGBoost ADULTS registrado (métricas + gráfico)
